This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Best practices for the real world

### Getting the most out of your models

#### Hyperparameter optimization

##### Using KerasTuner

In [ ]:
# PyTorch: KerasTuner has no native PyTorch equivalent. A common
# substitution is Optuna (`!pip install optuna -q`), but to keep this
# self-contained we implement a minimal manual grid search below using
# only torch/torchvision. (Run the install above if you prefer Optuna.)

In [ ]:
import torch
from torch import nn

# PyTorch: there is no KerasTuner. We represent a "hyperparameter set" as a
# plain dict and build the model from it. The Keras hp.Int("units", 16..64,
# step=16) and hp.Choice("optimizer", ["rmsprop","adam"]) become explicit
# choices we will iterate over in the search loop. We drop the final softmax
# because nn.CrossEntropyLoss expects raw logits.
def build_model(hp):
    units = hp["units"]
    model = nn.Sequential(
        nn.Linear(28 * 28, units),
        nn.ReLU(),
        nn.Linear(units, 10),
    )
    optimizer_name = hp["optimizer"]
    if optimizer_name == "rmsprop":
        optimizer = torch.optim.RMSprop(model.parameters())
    else:
        optimizer = torch.optim.Adam(model.parameters())
    loss_fn = nn.CrossEntropyLoss()
    return model, optimizer, loss_fn

In [ ]:
# PyTorch: KerasTuner's HyperModel becomes a small class that holds the fixed
# config (num_classes) and builds a model from a hyperparameter dict.
class SimpleMLP:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def build(self, hp):
        units = hp["units"]
        model = nn.Sequential(
            nn.Linear(28 * 28, units),
            nn.ReLU(),
            nn.Linear(units, self.num_classes),
        )
        optimizer_name = hp["optimizer"]
        if optimizer_name == "rmsprop":
            optimizer = torch.optim.RMSprop(model.parameters())
        else:
            optimizer = torch.optim.Adam(model.parameters())
        loss_fn = nn.CrossEntropyLoss()
        return model, optimizer, loss_fn

hypermodel = SimpleMLP(num_classes=10)

In [ ]:
# PyTorch: there is no kt.BayesianOptimization. We define the search space
# explicitly and a tiny grid-search "tuner" that trains each candidate and
# tracks the best by validation accuracy (the objective). executions_per_trial
# is kept to average noisy runs, like KerasTuner.
import itertools

search_space = {
    "units": [16, 32, 48, 64],
    "optimizer": ["rmsprop", "adam"],
}
max_trials = 20
executions_per_trial = 2

all_hp_combos = [
    dict(zip(search_space.keys(), values))
    for values in itertools.product(*search_space.values())
]
trial_hps = all_hp_combos[:max_trials]

In [ ]:
# PyTorch: print the search space we will iterate over.
print("Search space:")
for name, values in search_space.items():
    print(f"  {name}: {values}")

In [ ]:
# PyTorch: load MNIST via torchvision instead of keras.datasets and keep the
# same reshape/normalize preprocessing.
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
x_train, y_train = mnist_train.data.numpy(), mnist_train.targets.numpy()
x_test, y_test = mnist_test.data.numpy(), mnist_test.targets.numpy()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255
x_train_full = x_train[:]
y_train_full = y_train[:]
num_val_samples = 10000
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]

# PyTorch: tensors for training/validation.
x_train_t = torch.from_numpy(x_train)
y_train_t = torch.from_numpy(y_train).long()
x_val_t = torch.from_numpy(x_val)
y_val_t = torch.from_numpy(y_val).long()

# PyTorch: a compact training loop with EarlyStopping(patience) on val_loss,
# returning the per-epoch val_loss/val_accuracy history. Replaces model.fit
# plus keras.callbacks.EarlyStopping.
def train_model(model, optimizer, loss_fn, epochs, batch_size=128, patience=5):
    history = {"val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    wait = 0
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(len(x_train_t))
        for i in range(0, len(x_train_t), batch_size):
            idx = permutation[i : i + batch_size]
            inputs, targets = x_train_t[idx], y_train_t[idx]
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(x_val_t)
            val_loss = loss_fn(val_logits, y_val_t).item()
            val_acc = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    return history

# PyTorch: the search loop. For each hyperparameter set, train
# executions_per_trial times and average the best val accuracy.
trial_results = []
for hp in trial_hps:
    scores = []
    for _ in range(executions_per_trial):
        model, optimizer, loss_fn = build_model(hp)
        history = train_model(model, optimizer, loss_fn, epochs=100, patience=5)
        scores.append(max(history["val_accuracy"]))
    score = sum(scores) / len(scores)
    trial_results.append((score, hp))
    print(f"hp={hp} val_accuracy={score:.4f}")

In [ ]:
top_n = 4
# PyTorch: sort trials by the objective (val_accuracy) and take the best hp dicts.
best_hps = [
    hp for _, hp in sorted(trial_results, key=lambda r: r[0], reverse=True)[:top_n]
]

In [ ]:
def get_best_epoch(hp):
    model, optimizer, loss_fn = build_model(hp)
    # PyTorch: EarlyStopping(patience=10) is folded into train_model.
    history = train_model(model, optimizer, loss_fn, epochs=100, patience=10)
    val_loss_per_epoch = history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [ ]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model, optimizer, loss_fn = build_model(hp)
    # PyTorch: retrain on the full training set for a few more epochs, then
    # return the model. We use a simple loop (no validation/early stopping).
    x_full = torch.from_numpy(x_train_full)
    y_full = torch.from_numpy(y_train_full).long()
    batch_size = 128
    model.train()
    for epoch in range(int(best_epoch * 1.2)):
        permutation = torch.randperm(len(x_full))
        for i in range(0, len(x_full), batch_size):
            idx = permutation[i : i + batch_size]
            inputs, targets = x_full[idx], y_full[idx]
            predictions = model(inputs)
            loss = loss_fn(predictions, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model

# PyTorch: model.evaluate() becomes a manual no-grad loss/accuracy pass.
x_test_t = torch.from_numpy(x_test)
y_test_t = torch.from_numpy(y_test).long()
loss_fn = nn.CrossEntropyLoss()

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.eval()
    with torch.no_grad():
        logits = model(x_test_t)
        test_loss = loss_fn(logits, y_test_t).item()
        test_acc = (logits.argmax(dim=1) == y_test_t).float().mean().item()
    print(f"test_loss: {test_loss:.4f} test_acc: {test_acc:.4f}")
    best_models.append(model)

In [ ]:
# PyTorch: with no tuner object, the best models are simply the ones we just
# trained and collected above.
best_models = best_models[:top_n]

##### The art of crafting the right search space

##### The future of hyperparameter tuning: automated machine learning

#### Model ensembling

### Scaling up model training with multiple devices

#### Multi-GPU training

##### Data parallelism: Replicating your model on each GPU

##### Model parallelism: Splitting your model across multiple GPUs

#### Distributed training in practice

##### Getting your hands on two or more GPUs

##### Using data parallelism with JAX

##### Using model parallelism with JAX

###### The DeviceMesh API

###### The LayoutMap API

#### TPU training

##### Using step fusing to improve TPU utilization

### Speeding up training and inference with lower-precision computation

##### Understanding floating-point precision

##### Float16 inference

##### Mixed-precision training

##### Using loss scaling with mixed precision

##### Beyond mixed precision: float8 training

#### Faster inference with quantization

In [ ]:
# PyTorch: keras.ops.* map to torch.* ops.
import torch

x = torch.tensor([[0.1, 0.9], [1.2, -0.8]])
kernel = torch.tensor([[-0.1, -2.2], [1.1, 0.7]])

In [ ]:
def abs_max_quantize(value):
    # PyTorch: torch.* ops in place of keras.ops.*; cast via .to(dtype).
    abs_max = torch.max(torch.abs(value)).reshape(1)
    scale = 127 / (abs_max + 1e-7)
    scaled_value = value * scale
    scaled_value = torch.clip(torch.round(scaled_value), -127, 127)
    scaled_value = scaled_value.to(torch.int8)
    return scaled_value, scale

int_x, x_scale = abs_max_quantize(x)
int_kernel, kernel_scale = abs_max_quantize(kernel)

In [ ]:
# PyTorch: int8 matmul isn't supported on CPU, so promote to int32 first.
int_y = torch.matmul(int_x.to(torch.int32), int_kernel.to(torch.int32))
y = int_y.to(torch.float32) / (x_scale * kernel_scale)

In [0]:
y

In [ ]:
torch.matmul(x, kernel)